# Intercept trajectory viewer

Plots the logs that `intercept_controller_v2.py` (and `traditional_controller.py`)
write when `logs.save: true`. A run is a trial folder `logs/trajectory_logs/<name>_trial_<N>/`
holding `<name>_pursuer.csv` and `<name>_evader.csv`, one row per control tick with
`t_rel,state_stamp,x,y,z,vx,vy,vz` (world frame).

Set `RUN` to a trial folder name (default: the newest run) and run all cells. An **x** marks the
interception: the first tick where the drones are within `CAPTURE_RADIUS`, the same
success rule as training. Set `SHOW_STATE_JUMPS = True` to also mark ticks where a
drone's position jumps more than `STATE_JUMP` in one tick (faster than the airframe can
fly, e.g. a mocap rigid-body swap), since distances there are not trustworthy.
Needs only `numpy` and `matplotlib >= 3.7`.

In [ ]:
from pathlib import Path
import csv

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D

# %matplotlib widget          # uncomment (needs ipympl) to rotate / zoom the 3D view

LOG_DIR = Path('logs/trajectory_logs')  # relative to this notebook (scripts/deploy/)
RUN = None                         # trial folder, e.g. 'intercept_trial_1'; None = newest run
COMPARE = None                     # list of trial folders to overlay; None = all runs (max 8)
CAPTURE_RADIUS = 0.10              # m, success_radius_eval in cfg/task/Intercept.yaml
MIN_ALTITUDE = 0.15                # m, controller.min_altitude in config_v2.yaml
STATE_JUMP = 0.20                  # m; a bigger position change in one tick is a state jump
SHOW_STATE_JUMPS = False           # True = mark state jumps (likely tracking glitches)
VIEW_3D = (22, -60)                # (elevation, azimuth) of the 3D view in degrees
THEME = 'light'                    # 'light' | 'dark'
SAVE_DIR = None                    # e.g. Path('logs/plots') to also save PNGs

In [ ]:
# Validated categorical palette: slot 1 = pursuer, slot 2 = evader, further slots
# only for the multi-run comparison (assigned in order, never cycled).
THEMES = {
    'light': dict(surface='#fcfcfb', ink='#0b0b0b', ink2='#52514e', muted='#898781',
                  grid='#e1e0d9', axis='#c3c2b7',
                  series=['#2a78d6', '#eb6834', '#1baf7a', '#eda100',
                          '#e87ba4', '#008300', '#4a3aa7', '#e34948']),
    'dark': dict(surface='#1a1a19', ink='#ffffff', ink2='#c3c2b7', muted='#898781',
                 grid='#2c2c2a', axis='#383835',
                 series=['#3987e5', '#d95926', '#199e70', '#c98500',
                         '#d55181', '#008300', '#9085e9', '#e66767']),
}
C = THEMES[THEME]
PURSUER, EVADER = C['series'][0], C['series'][1]
RELATIVE = C['ink2']  # quantities between the two drones (separation, closing speed)
GLITCH = '#d03b3b'    # status "critical" (both themes); always a triangle marker + a label

plt.rcParams.update({
    'figure.facecolor': C['surface'], 'axes.facecolor': C['surface'],
    'savefig.facecolor': C['surface'], 'text.color': C['ink'],
    'axes.edgecolor': C['axis'], 'axes.linewidth': 0.8,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.labelcolor': C['ink2'], 'axes.labelsize': 9,
    'axes.titlecolor': C['ink'], 'axes.titlesize': 10, 'axes.titleweight': 'semibold',
    'axes.titlelocation': 'left',
    'axes.grid': True, 'grid.color': C['grid'], 'grid.linewidth': 0.6, 'grid.linestyle': '-',
    'xtick.color': C['axis'], 'ytick.color': C['axis'],
    'xtick.labelcolor': C['muted'], 'ytick.labelcolor': C['muted'],
    'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'legend.frameon': False, 'legend.fontsize': 9, 'legend.labelcolor': C['ink2'],
    'lines.linewidth': 1.6, 'lines.solid_capstyle': 'round',
})
MARK = dict(ls='', ms=7, mec=C['surface'], mew=1.5, zorder=4)  # markers with a surface ring
HALO = [pe.withStroke(linewidth=5, foreground=C['surface'])]  # surface ring around the x
INTERCEPT_MARK = dict(ls='', marker='x', ms=11, mew=2.4, color=C['ink'], zorder=6,
                      path_effects=HALO)
JUMP_MARK = dict(ls='', marker='^', ms=7, mfc=GLITCH, mec=C['surface'], mew=1, zorder=5)

## Loading

The controller logs both drones on the same tick, so rows line up one-to-one. Logs
that don't (e.g. hand-made files) get the evader resampled onto the pursuer's ticks.
Time is shifted so `t = 0` is the first policy tick.

In [ ]:
def find_runs(log_dir=LOG_DIR):
    # Trial folders with both a pursuer and an evader CSV, oldest first.
    runs = []
    for path in log_dir.glob('*/*_pursuer.csv'):
        name = path.name[:-len('_pursuer.csv')]
        if (path.parent / f'{name}_evader.csv').is_file():
            runs.append((path.stat().st_mtime, path.parent.name))
    return [trial for _, trial in sorted(runs)]


def load_csv(path):
    with open(path, newline='', encoding='utf-8') as handle:
        rows = list(csv.DictReader(handle))
    if len(rows) < 2:
        raise ValueError(f'{path} has {len(rows)} data rows; need at least 2.')
    cols = lambda *keys: np.array([[float(row[k]) for k in keys] for row in rows])
    return {'t': cols('t_rel')[:, 0], 'stamp': cols('state_stamp')[:, 0],
            'pos': cols('x', 'y', 'z'), 'vel': cols('vx', 'vy', 'vz')}


def load_run(prefix, log_dir=LOG_DIR):
    # prefix: trial folder name, e.g. 'intercept_trial_1'.
    pursuer_csv = next((log_dir / prefix).glob('*_pursuer.csv'))
    p = load_csv(pursuer_csv)
    e = load_csv(pursuer_csv.with_name(pursuer_csv.name[:-len('_pursuer.csv')] + '_evader.csv'))
    t = p['t']
    if len(e['t']) != len(t) or not np.allclose(e['t'], t):
        resample = lambda v: np.column_stack([np.interp(t, e['t'], v[:, k]) for k in range(3)])
        e = {'t': t, 'stamp': np.interp(t, e['t'], e['stamp']),
             'pos': resample(e['pos']), 'vel': resample(e['vel'])}
    rel_pos = e['pos'] - p['pos']
    rel_vel = e['vel'] - p['vel']
    dist = np.linalg.norm(rel_pos, axis=1)
    closing = -np.einsum('ij,ij->i', rel_pos, rel_vel) / np.maximum(dist, 1e-6)
    step = lambda A: np.r_[0.0, np.linalg.norm(np.diff(A, axis=0), axis=1)]
    jumps = {role: np.flatnonzero(step(drone['pos']) > STATE_JUMP)
             for role, drone in (('pursuer', p), ('evader', e))}
    inside = np.flatnonzero(dist <= CAPTURE_RADIUS)
    return {'prefix': prefix, 't': t - t[0], 'pursuer': p, 'evader': e,
            'dist': dist, 'closing': closing, 'i_min': int(np.argmin(dist)),
            'i_capture': int(inside[0]) if inside.size else None,  # interception tick
            'jumps': jumps, 'jump_ticks': np.union1d(*jumps.values()).astype(int)}


def event_tick(run):
    # The tick the plots highlight: the interception, else the closest approach.
    return run['i_min'] if run['i_capture'] is None else run['i_capture']


def summarize(run):
    d, t, i, ic = run['dist'], run['t'], run['i_min'], run['i_capture']
    vmax = lambda drone: np.linalg.norm(drone['vel'], axis=1).max()
    row = {
        'run': run['prefix'],
        'duration_s': t[-1],
        'start_dist_m': d[0],
        'min_dist_m': d[i],
        't_min_dist_s': t[i],
        't_capture_s': None if ic is None else t[ic],
        'pursuer_vmax': vmax(run['pursuer']),
        'evader_vmax': vmax(run['evader']),
        'pursuer_zmin': run['pursuer']['pos'][:, 2].min(),
    }
    if SHOW_STATE_JUMPS:
        row['state_jumps'] = len(run['jump_ticks'])
    return row


def print_table(rows):
    # Plain-text table: every value the charts show, readable without them.
    def fmt(v):
        if v is None:
            return '-'
        return f'{v:.2f}' if isinstance(v, (float, np.floating)) else str(v)
    cols = list(rows[0])
    cells = [[fmt(row[c]) for c in cols] for row in rows]
    widths = [max(len(c), *(len(r[k]) for r in cells)) for k, c in enumerate(cols)]
    align = lambda k, s: s.ljust(widths[k]) if k == 0 else s.rjust(widths[k])
    print('  '.join(align(k, c) for k, c in enumerate(cols)))
    print('  '.join('-' * w for w in widths))
    for r in cells:
        print('  '.join(align(k, s) for k, s in enumerate(r)))


def save(fig, name):
    if SAVE_DIR is not None:
        Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
        fig.savefig(Path(SAVE_DIR) / f'{name}.png', dpi=200)

In [ ]:
if not LOG_DIR.is_dir():  # notebook started from the repo root instead of scripts/deploy/
    LOG_DIR = Path('scripts/deploy') / LOG_DIR
runs = find_runs(LOG_DIR)
if not runs:
    raise FileNotFoundError(f'No <name>_trial_<N>/ folders with <name>_pursuer.csv + <name>_evader.csv in {LOG_DIR.resolve()}')
print(f'{len(runs)} run(s) in {LOG_DIR.resolve()}: {", ".join(runs)}\n')

run = load_run(RUN or runs[-1], LOG_DIR)
print_table([summarize(run)])
for role, idx in run['jumps'].items():
    if SHOW_STATE_JUMPS and idx.size:
        times = ', '.join(f'{run["t"][k]:.2f}' for k in idx)
        print(f'\nWarning: {role} position jumps > {STATE_JUMP:.2f} m in one tick at t = {times} s. '
              'Likely a tracking glitch (mocap rigid-body swap / EKF reset).')

## Flight paths

3D view plus three projections at equal scale. Circles mark the start, squares the
end, and the x the interception (midpoint of the two drones at that tick). A run with
no interception gets a gray segment joining the drones at their closest approach
instead. With `SHOW_STATE_JUMPS`, red triangles mark where a drone's position jumped.

In [ ]:
def _draw_paths(ax, run, dims):
    P, E = run['pursuer']['pos'], run['evader']['pos']
    for role, A, color in (('pursuer', P, PURSUER), ('evader', E, EVADER)):
        ax.plot(*(A[:, d] for d in dims), color=color)
        ax.plot(*(A[:1, d] for d in dims), marker='o', mfc=color, **MARK)
        ax.plot(*(A[-1:, d] for d in dims), marker='s', mfc=color, **MARK)
        jumps = run['jumps'][role]
        if SHOW_STATE_JUMPS and jumps.size:
            ax.plot(*(A[jumps, d] for d in dims), **JUMP_MARK)
    i = event_tick(run)
    if run['i_capture'] is not None:
        mid = (P[i] + E[i]) / 2
        ax.plot(*([mid[d]] for d in dims), **INTERCEPT_MARK)
    else:
        ax.plot(*([P[i, d], E[i, d]] for d in dims), color=RELATIVE, lw=1.2, zorder=3)


def _reference(ax, y, label, side='right'):
    # Thin solid reference line with a direct label at one end.
    ax.axhline(y, color=C['muted'], lw=0.8, zorder=1)
    right = side == 'right'
    ax.annotate(label, (1 if right else 0, y), xycoords=('axes fraction', 'data'),
                xytext=(-2 if right else 2, 3), textcoords='offset points',
                ha=side, va='bottom', color=C['muted'], fontsize=8)


def _headline(run):
    i = event_tick(run)
    d, t = run['dist'][i], run['t'][i]
    if run['i_capture'] is not None:
        text = (f"{run['prefix']}: interception at t = {t:.2f} s "
                f"(separation {d:.2f} m, capture radius {CAPTURE_RADIUS:.2f} m)")
    else:
        text = (f"{run['prefix']}: no interception, closest approach {d:.2f} m "
                f"at t = {t:.2f} s (capture radius {CAPTURE_RADIUS:.2f} m)")
    if SHOW_STATE_JUMPS and np.any(np.abs(run['jump_ticks'] - i) <= 2):
        text += ', at a state jump: check tracking'
    return text


def _event_handle(run):
    if run['i_capture'] is not None:
        return Line2D([], [], label='Interception', **INTERCEPT_MARK)
    return Line2D([], [], color=RELATIVE, lw=1.2, label='Closest approach')


def _figure_legend(fig, run, handles):
    # Headline as the legend title: constrained layout can't stack a suptitle above
    # an 'outside' legend, so both live in one artist.
    if SHOW_STATE_JUMPS and run['jump_ticks'].size:
        handles.append(Line2D([], [], label=f'State jump > {STATE_JUMP:.2f} m/tick', **JUMP_MARK))
    fig.legend(handles=handles, loc='outside upper left', ncol=len(handles),
               title=_headline(run), alignment='left',
               title_fontproperties={'size': 11, 'weight': 'semibold'})


def plot_paths(run):
    P, E = run['pursuer']['pos'], run['evader']['pos']
    fig = plt.figure(figsize=(12, 9.5), layout='constrained')
    gs = fig.add_gridspec(2, 2)

    ax3 = fig.add_subplot(gs[0, 0], projection='3d')
    pts = np.vstack([P, E])
    lo, hi = pts.min(0), pts.max(0)
    lo[2] = min(lo[2], 0.0)  # keep the floor in view
    pad = 0.05 * max((hi - lo).max(), 0.2)
    lo, hi = lo - pad, hi + pad
    for A, color in ((P, PURSUER), (E, EVADER)):  # floor shadows as a depth cue
        ax3.plot(A[:, 0], A[:, 1], np.full(len(A), lo[2]), color=color, lw=1, alpha=0.25)
    _draw_paths(ax3, run, (0, 1, 2))
    ax3.set(xlim=(lo[0], hi[0]), ylim=(lo[1], hi[1]), zlim=(lo[2], hi[2]),
            xlabel='x (m)', ylabel='y (m)', zlabel='z (m)')
    ax3.set_box_aspect(hi - lo)  # equal scale on all three axes
    ax3.view_init(*VIEW_3D)
    ax3.set_title('3D')
    for axis in (ax3.xaxis, ax3.yaxis, ax3.zaxis):
        axis.pane.fill = False
        axis.pane.set_edgecolor(C['grid'])

    views = [(gs[0, 1], (0, 1), 'Top view (x-y)'),
             (gs[1, 0], (0, 2), 'Side view (x-z)'),
             (gs[1, 1], (1, 2), 'Front view (y-z)')]
    for spec, dims, title in views:
        ax = fig.add_subplot(spec)
        _draw_paths(ax, run, dims)
        if dims[1] == 2:
            _reference(ax, MIN_ALTITUDE, 'min altitude')
        ax.set_aspect('equal', adjustable='datalim')
        ax.set(title=title, xlabel='xyz'[dims[0]] + ' (m)', ylabel='xyz'[dims[1]] + ' (m)')

    handles = [
        Line2D([], [], color=PURSUER, label='Pursuer'),
        Line2D([], [], color=EVADER, label='Evader'),
        Line2D([], [], marker='o', mfc=C['muted'], label='Start', **MARK),
        Line2D([], [], marker='s', mfc=C['muted'], label='End', **MARK),
        _event_handle(run),
    ]
    _figure_legend(fig, run, handles)
    return fig


fig = plot_paths(run)
save(fig, f"{run['prefix']}_paths")
plt.show()

## Over time

Closing speed is the rate the separation shrinks (positive = closing in). The thin
vertical line marks the interception (or, without one, the closest approach) in every
panel.

In [ ]:
def plot_timeseries(run):
    t, P, E, i = run['t'], run['pursuer'], run['evader'], event_tick(run)
    captured = run['i_capture'] is not None
    fig, axes = plt.subplots(3, 2, figsize=(12, 8), sharex=True, layout='constrained')
    (ax_d, ax_x), (ax_c, ax_y), (ax_s, ax_z) = axes

    d_i = run['dist'][i]
    ax_d.plot(t, run['dist'], color=RELATIVE)
    if captured:
        ax_d.plot(t[i], d_i, **INTERCEPT_MARK)
    else:
        ax_d.plot(t[i], d_i, marker='o', mfc=RELATIVE, **MARK)
    _reference(ax_d, CAPTURE_RADIUS, f'capture radius {CAPTURE_RADIUS:.2f} m',
               side='left' if t[i] > 0.5 * t[-1] else 'right')  # keep clear of the marker
    if SHOW_STATE_JUMPS and run['jump_ticks'].size:
        ax_d.plot(t[run['jump_ticks']], run['dist'][run['jump_ticks']], **JUMP_MARK)
    ax_d.set_ylim(bottom=0)
    ax_d.set_title(f"Separation (m), {'interception' if captured else 'closest'} "
                   f"{d_i:.2f} m at t = {t[i]:.2f} s")

    ax_c.plot(t, run['closing'], color=RELATIVE)
    ax_c.axhline(0, color=C['axis'], lw=0.8, zorder=1)
    ax_c.set_title('Closing speed (m/s)')

    for drone, color in ((P, PURSUER), (E, EVADER)):
        ax_s.plot(t, np.linalg.norm(drone['vel'], axis=1), color=color)
        for k, ax in enumerate((ax_x, ax_y, ax_z)):
            ax.plot(t, drone['pos'][:, k], color=color)
    ax_s.set_ylim(bottom=0)
    ax_s.set_title('Speed (m/s)')
    ax_x.set_title('x (m)')
    ax_y.set_title('y (m)')
    ax_z.set_title('z, altitude (m)')
    _reference(ax_z, MIN_ALTITUDE, f'min altitude {MIN_ALTITUDE:.2f} m')

    for ax in axes.flat:
        ax.axvline(t[i], color=C['muted'], lw=0.8, zorder=0)
    for ax in axes[-1]:
        ax.set_xlabel('time since policy start (s)')

    _figure_legend(fig, run, [
        Line2D([], [], color=PURSUER, label='Pursuer'),
        Line2D([], [], color=EVADER, label='Evader'),
        Line2D([], [], color=RELATIVE, label='Pursuer to evader'),
        Line2D([], [], color=C['muted'], lw=0.8,
               label='Interception time' if captured else 'Closest approach time'),
    ])
    return fig


fig = plot_timeseries(run)
save(fig, f"{run['prefix']}_timeseries")
plt.show()

## Compare runs

Separation over time for several runs on one axis. An x marks each run's interception;
a dot marks the closest approach of runs that never got inside the capture radius.
Colors follow the order of `COMPARE` (or of the runs list), up to 8 runs.

In [ ]:
def plot_comparison(prefixes, log_dir=LOG_DIR):
    if len(prefixes) > len(C['series']):
        print(f'{len(prefixes)} runs; showing the newest {len(C["series"])}. Set COMPARE to pick.')
        prefixes = prefixes[-len(C['series']):]
    loaded = []
    for prefix in prefixes:
        try:
            loaded.append(load_run(prefix, log_dir))
        except ValueError as exc:
            print(f'Skipping {prefix}: {exc}')

    fig, ax = plt.subplots(figsize=(12, 4.5), layout='constrained')
    label_ends = len(loaded) <= 4
    shapes = {}
    for r, color in zip(loaded, C['series']):
        i = event_tick(r)
        ax.plot(r['t'], r['dist'], color=color, label=r['prefix'])
        if r['i_capture'] is not None:
            ax.plot(r['t'][i], r['dist'][i], **{**INTERCEPT_MARK, 'color': color})
            shapes['x'] = Line2D([], [], label='Interception', **{**INTERCEPT_MARK, 'color': C['muted']})
        else:
            ax.plot(r['t'][i], r['dist'][i], marker='o', mfc=color, **MARK)
            shapes['o'] = Line2D([], [], marker='o', mfc=C['muted'],
                                 label='Closest approach (no interception)', **MARK)
        if label_ends:
            ax.annotate(r['prefix'], (r['t'][-1], r['dist'][-1]), xytext=(5, 0),
                        textcoords='offset points', va='center', color=C['ink2'], fontsize=9)
    if label_ends:  # room for the end labels
        ax.set_xlim(right=max(r['t'][-1] for r in loaded) * 1.15)
    _reference(ax, CAPTURE_RADIUS, f'capture radius {CAPTURE_RADIUS:.2f} m')
    ax.set_ylim(bottom=0)
    ax.set(title='Separation by run (m)', xlabel='time since policy start (s)')
    run_handles, _ = ax.get_legend_handles_labels()
    ax.legend(handles=run_handles + [shapes[k] for k in ('x', 'o') if k in shapes],
              loc='upper right')
    print_table([summarize(r) for r in loaded])
    return fig


fig = plot_comparison(COMPARE or runs)
save(fig, 'comparison')
plt.show()